In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('../data/raw/cleaned_data_v2.csv')

In [4]:
df.head()

,bhk,type,locality,area,price,price_unit,region,status,age,final_price
0,3,Apartment,Lak And Hanware The Residency Tower,685,2.50,Cr,Andheri West,Ready to move,New,250.00
1,2,Apartment,Radheya Sai Enclave Building No 2,640,52.51,L,Naigaon East,Under Construction,New,52.51
2,2,Apartment,Romell Serene,610,1.73,Cr,Borivali West,Under Construction,New,173.00
3,2,Apartment,Soundlines Codename Urban Rainforest,876,59.98,L,Panvel,Under Construction,New,59.98
4,2,Apartment,Origin Oriana,659,94.11,L,Mira Road East,Under Construction,New,94.11


In [5]:
df.shape

(76038, 10)

In [6]:
df.columns.tolist()

['bhk',
 'type',
 'locality',
 'area',
 'price',
 'price_unit',
 'region',
 'status',
 'age',
 'final_price']

In [7]:
df[["price", "price_unit", "final_price"]].head(10)

,price,price_unit,final_price
0,2.50,Cr,250.00
1,52.51,L,52.51
2,1.73,Cr,173.00
3,59.98,L,59.98
4,94.11,L,94.11
5,3.30,Cr,330.00
6,1.99,Cr,199.00
7,2.89,Cr,289.00
8,29.84,L,29.84
9,2.72,Cr,272.00


In [8]:
print(df["price_unit"].value_counts(dropna=False))
print("\nMissing values:")
print(df[["price", "price_unit", "final_price"]].isnull().sum())

price_unit
Cr    40981
L     35057
Name: count, dtype: int64

Missing values:
price          0
price_unit     0
final_price    0
dtype: int64


In [9]:
# Check whether final_price matches price converted to lakhs
expected_price = df["price"].where(
    df["price_unit"].eq("L"),
    df["price"] * 100
)

difference = (df["final_price"] - expected_price).abs()

print("Maximum conversion difference:", difference.max())
print("Rows with inconsistent prices:", (difference > 0.01).sum())
print("\nFinal price summary (lakhs):")
print(df["final_price"].describe().round(2))

Maximum conversion difference: 4.547473508864641e-13
Rows with inconsistent prices: 0

Final price summary (lakhs):
count    76038.00
mean       168.42
std        217.67
min          4.49
25%         64.00
50%        110.00
75%        194.00
max       6000.00
Name: final_price, dtype: float64


In [10]:
df.dtypes

bhk              int64
type            object
locality        object
area             int64
price          float64
price_unit      object
region          object
status          object
age             object
final_price    float64
dtype: object

In [11]:
df.isnull().sum()

bhk            0
type           0
locality       0
area           0
price          0
price_unit     0
region         0
status         0
age            0
final_price    0
dtype: int64

In [12]:
df.duplicated().sum()

np.int64(20312)

In [13]:
print("\nUNIQUE VALUES IN CATEGORICAL COLUMNS")
for col in ["type", "region", "status", "age"]:
    print(f"\n{col}:")
    print(df[col].value_counts().head(10))


UNIQUE VALUES IN CATEGORICAL COLUMNS

type:
type
Apartment            74854
Studio Apartment       882
Villa                  226
Independent House       73
Penthouse                3
Name: count, dtype: int64

region:
region
Thane West        14868
Mira Road East     9902
Dombivali          3041
Kandivali East     2568
Kharghar           2362
Andheri West       1842
Kalyan West        1815
Virar              1557
Mulund West        1526
Panvel             1474
Name: count, dtype: int64

status:
status
Ready to move         44982
Under Construction    31056
Name: count, dtype: int64

age:
age
New        38072
Resale     23357
Unknown    14609
Name: count, dtype: int64


In [14]:
# Features (X): information used to predict price
feature_columns = [
    "bhk",
    "type",
    "locality",
    "area",
    "region",
    "status",
    "age"
]

X = df[feature_columns]
y = df["final_price"]

In [15]:
X.shape

(76038, 7)

In [16]:
y.shape

(76038,)

In [17]:
X.head()

,bhk,type,locality,area,region,status,age
0,3,Apartment,Lak And Hanware The Residency Tower,685,Andheri West,Ready to move,New
1,2,Apartment,Radheya Sai Enclave Building No 2,640,Naigaon East,Under Construction,New
2,2,Apartment,Romell Serene,610,Borivali West,Under Construction,New
3,2,Apartment,Soundlines Codename Urban Rainforest,876,Panvel,Under Construction,New
4,2,Apartment,Origin Oriana,659,Mira Road East,Under Construction,New


In [18]:
y.head()

0    250.00
1     52.51
2    173.00
3     59.98
4     94.11
Name: final_price, dtype: float64

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
X_train, X_test, y_train, y_test = train_test_split (
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

In [21]:
X_train.shape, y_train.shape

((60830, 7), (60830,))

In [22]:
X_test.shape, y_test.shape

((15208, 7), (15208,))

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

categorical_features = [
    "type", "locality", "region", "status", "age"
]

numeric_features = ["bhk", "area"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

print("Model training complete!")

Model training complete!


In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = model.predict(X_test)

In [25]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

In [26]:
print(f"MAE: ₹{mae:.2f} lakhs") 
print(f"MSE: ₹{mse:.2f} lakhs") 
print(f"RMSE: ₹{rmse:.2f} lakhs") 

MAE: ₹35.13 lakhs
MSE: ₹6351.50 lakhs
RMSE: ₹79.70 lakhs


In [27]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

baseline_mae = mean_absolute_error(
    y_test, baseline_predictions
)

print(f"Baseline MAE: ₹{baseline_mae:.2f} lakhs")
print(f"Linear Regression MAE: ₹{mae:.2f} lakhs")

Baseline MAE: ₹103.54 lakhs
Linear Regression MAE: ₹35.13 lakhs


In [28]:
# Create a table of actual and predicted prices
results = pd.DataFrame({
    "actual_price": y_test.reset_index(drop=True),
    "predicted_price": y_pred
})

# Calculate the signed prediction error
results["residual"] = (
    results["actual_price"] - results["predicted_price"]
)

# Calculate the size of the error, regardless of direction
results["absolute_error"] = results["residual"].abs()

# Display the 10 predictions with the largest errors
display(
    results.nlargest(10, "absolute_error")
)

,actual_price,predicted_price,residual,absolute_error
10291,5000.0,2106.142614,2893.857386,2893.857386
12306,5555.0,2731.787796,2823.212204,2823.212204
15108,3400.0,1600.264107,1799.735893,1799.735893
13691,3400.0,1762.427424,1637.572576,1637.572576
3694,3300.0,1748.657411,1551.342589,1551.342589
10855,5000.0,3586.557256,1413.442744,1413.442744
6527,2500.0,1335.410920,1164.589080,1164.589080
8895,2500.0,3639.000092,-1139.000092,1139.000092
7439,925.0,1992.427359,-1067.427359,1067.427359
7846,925.0,1992.427359,-1067.427359,1067.427359


In [29]:
# Get the original features for the 10 worst predictions
worst_indices = results.nlargest(10, "absolute_error").index

worst_properties = X_test.reset_index(drop=True).iloc[worst_indices].copy()

# Add actual and predicted prices to the feature table
worst_properties["actual_price"] = y_test.reset_index(drop=True).iloc[worst_indices].values
worst_properties["predicted_price"] = y_pred[worst_indices]

display(worst_properties)

,bhk,type,locality,area,region,status,age,actual_price,predicted_price
10291,8,Apartment,Rustomjee Elements,7500,Andheri West,Ready to move,New,5000.0,2106.142614
12306,3,Apartment,Rustomjee Paramount Wing F,10893,Santacruz West,Under Construction,New,5555.0,2731.787796
15108,5,Apartment,Lodha The World Towers World One Tier III And ...,6000,Lower Parel,Ready to move,New,3400.0,1600.264107
13691,9,Apartment,Reputed Builder Varsha Apartments,6350,Juhu,Ready to move,Resale,3400.0,1762.427424
3694,10,Apartment,Reputed Builder Varsha Apartments,6200,Juhu,Ready to move,Resale,3300.0,1748.657411
10855,5,Apartment,Provenance Four Seasons Private Residences,7000,Worli,Under Construction,New,5000.0,3586.557256
6527,8,Apartment,Reputed Builder Atlantis,7800,Kharghar,Ready to move,Resale,2500.0,1335.410920
8895,6,Apartment,SD The Imperial Edge,10000,Tardeo,Ready to move,Unknown,2500.0,3639.000092
7439,3,Apartment,Lodha World One,3000,Lower Parel,Ready to move,Resale,925.0,1992.427359
7846,3,Apartment,Lodha World One,3000,Lower Parel,Ready to move,Resale,925.0,1992.427359


In [30]:
# Count how often each feature combination appears
feature_columns = [
    "bhk", "type", "locality", "area", "region", "status", "age"
]

duplicate_features = df.duplicated(
    subset=feature_columns,
    keep=False
)

# Show rows that share the same feature combination
duplicate_rows = df.loc[duplicate_features].sort_values(
    by=feature_columns
)

print("Rows involved in duplicate feature combinations:",
      len(duplicate_rows))

display(duplicate_rows.head(20))

Rows involved in duplicate feature combinations: 41336


,bhk,type,locality,area,price,price_unit,region,status,age,final_price
13935,1,Apartment,Asha Nagar,650,1.05,Cr,Mulund West,Ready to move,Resale,105.0
56072,1,Apartment,Asha Nagar,650,1.00,Cr,Mulund West,Ready to move,Resale,100.0
33348,1,Apartment,Dwarkesh Dham C H S,600,90.00,L,Dahisar West,Ready to move,Resale,90.0
33548,1,Apartment,Dwarkesh Dham C H S,600,90.00,L,Dahisar West,Ready to move,Resale,90.0
33627,1,Apartment,Dwarkesh Dham C H S,600,90.00,L,Dahisar West,Ready to move,Resale,90.0
34154,1,Apartment,Dwarkesh Dham C H S,600,90.00,L,Dahisar West,Ready to move,Resale,90.0
52360,1,Apartment,J B Heights,685,42.00,L,Karanjade,Ready to move,Resale,42.0
67016,1,Apartment,J B Heights,685,38.00,L,Karanjade,Ready to move,Resale,38.0
4894,1,Apartment,Radhe Krishna Heights,450,21.00,L,Karjat,Ready to move,New,21.0
5189,1,Apartment,Radhe Krishna Heights,450,22.90,L,Karjat,Ready to move,New,22.9
